# BMD-45 Finale (joint polish + full-val report)

Closes the 8-loop chain (0.7949 → 0.8293). One session:
1. Rebuild ALL folders (8 train + 3 val, ~46k images) via the chunked
   pipeline — per-folder checkpoints, ~1.5-2h prep.
2. Polish: 5 epochs from loop-8 `best.pt` at low LR over the full pool.
3. Report on the FULL official val (~10k images), not the 3.4k anchor.
4. Package `india-yolov8n-final` + output pack zip for the workspace.

Expected: +0.002–0.005 overall and a smoother tail — polish, not
breakthrough. If overall drops >0.01 vs 0.8293, the polish hurt: keep
the loop-8 registry as the winner and say so in the handoff.

## Runtime
T4 GPU, one session (~4-5h total). Tab focused, cells staged.

## 0. Params — edit once

In [ ]:
# ---- edit ----------------------------------------------------------
PREV_WEIGHTS = "/content/best_f007.pt"  # loop-8 best.pt, uploaded below
EPOCHS = 5
LR0 = 0.002  # low-LR polish; do not raise without evidence
# ---- stable --------------------------------------------------------
HF_REPO = "iisc-aim/BMD-45"
HF_TRAIN = "BMD-45-Train"
HF_VAL = "BMD-45-Val"
TRAIN_FOLDERS = ["images_%03d" % i for i in range(8)]
VAL_FOLDERS = ["images_%03d" % i for i in range(3)]
ROOT = "/tmp/bmd_finale"

print('init:', PREV_WEIGHTS, '| epochs:', EPOCHS, '| lr0:', LR0)

## 1. Setup

Uploads needed (file browser): this notebook + `best_f007.pt` (~6 MB).
Pinned deps, HF auth, ~55 GB disk guard (peak = one PNG folder + all JPGs).

In [ ]:
!pip install -q ultralytics onnx onnxruntime huggingface_hub

import os
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
assert os.path.isfile(PREV_WEIGHTS), 'upload best_f007.pt first: ' + PREV_WEIGHTS

try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        print('HF auth: token loaded')
    else:
        print('HF auth: anonymous (slower)')
except Exception as e:
    print('HF auth: userdata unavailable (%s), anonymous' % e)

import shutil
free_gb = shutil.disk_usage('/tmp').free / 1e9
print('/tmp free: %.1f GB' % free_gb)
assert free_gb > 55, 'need ~55 GB free for all 11 folders; free space and retry'
os.makedirs(ROOT, exist_ok=True)


## 2. Prep-all (idempotent, per-folder checkpoints)

Same verified chunked logic as the loop notebook: JSONs first, then one
folder at a time (download → transcode → delete PNGs). ~1.5-2h for 11 folders.
Re-running resumes at the first missing folder.

In [ ]:
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
import glob, json, os, shutil
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
import cv2
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import snapshot_download
import time as _time, random as _random

CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]
ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "two_wheeler": "motorcycle", "two-wheeler": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle", "cycle": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
    "three-wheeler": "auto",
    "sedan": "car", "hatchback": "car", "suv": "car", "muv": "car",
    "minibus": "bus", "mini_bus": "bus", "van": "bus",
    "tempo_traveller": "bus", "tempo": "bus",
    "lcv": "truck",
}

def class_index(name):
    n = (name or "").lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return CLASSES.index(m) if m in CLASSES else None

def _transient(exc):
    msg = str(exc)
    return ("429" in msg or "Too Many Requests" in msg
            or 'ConnectionError' in type(exc).__name__
            or 'Timeout' in type(exc).__name__)

DL_WORKERS = 4

def _dl(label, tries=8, base_wait=30.0, **kw):
    for attempt in range(1, tries + 1):
        try:
            return snapshot_download(**kw)
        except Exception as e:
            if not _transient(e) or attempt == tries:
                print('[%s] giving up after %d attempts: %s' % (label, attempt, e))
                raise
            wait = base_wait * (1.6 ** (attempt - 1)) + _random.uniform(0, 10)
            print('[%s] transient, retry %d/%d in %.0fs' % (label, attempt + 1, tries, wait))
            _time.sleep(wait)

meta = _dl('meta-json', repo_id=HF_REPO, repo_type='dataset',
             allow_patterns=["BMD-45-Train/*.json", "BMD-45-Val/*.json"],
             max_workers=DL_WORKERS)
TABLE = {}
for subset, split in ((HF_TRAIN, 'train'), (HF_VAL, 'val')):
    afs = sorted(Path(meta, subset).rglob('*annotations*.json'))
    assert afs, 'no COCO JSON for ' + subset
    data = json.loads(afs[0].read_text(encoding='utf-8'))
    cats = {c['id']: c['name'] for c in data.get('categories', [])}
    imgs = {im['id']: im for im in data.get('images', [])}
    n_keep = n_drop = 0
    for a in data.get('annotations', []):
        cname = cats.get(a.get('category_id'))
        idx = class_index(cname) if cname else None
        if idx is None:
            n_drop += 1
            continue
        im = imgs.get(a.get('image_id'))
        if im is None:
            continue
        iw, ih = im.get('width') or 1, im.get('height') or 1
        x, y, w, h = a['bbox']
        cx, cy = (x + w / 2) / iw, (y + h / 2) / ih
        stem = Path(str(im.get('file_name', ''))).stem
        line = str(idx) + ' %.6f %.6f %.6f %.6f' % (cx, cy, w / iw, h / ih)
        TABLE.setdefault((split, stem), []).append(line)
        n_keep += 1
    print('%s: kept %d boxes, dropped %d' % (subset, n_keep, n_drop))
assert TABLE, 'empty label table'

def ensure_folder(subset, fld, split):
    have = {p.stem for p in Path(ROOT, 'images', split).glob('*.jpg')} if Path(ROOT, 'images', split).is_dir() else set()
    _anchor = ROOT if os.path.isdir(ROOT) else os.path.dirname(ROOT)
    free_gb = shutil.disk_usage(_anchor).free / 1e9
    assert free_gb > 18, ('only %.1f GB free before %s/%s: wipe '
                          '/root/.cache/huggingface and re-run prep' % (free_gb, subset, fld))
    d = _dl(subset + '/' + fld, repo_id=HF_REPO, repo_type='dataset',
            allow_patterns=[subset + '/' + fld + '/**'],
            max_workers=DL_WORKERS)
    src_dir = Path(d) / subset / fld
    for sub in ('images', 'labels'):
        os.makedirs(os.path.join(ROOT, sub, split), exist_ok=True)
    cands = [p for p in sorted(src_dir.glob('*.png'))
             if TABLE.get((split, p.stem)) and p.stem not in have]

    def _one(png):
        img = cv2.imread(str(png))
        assert img is not None, 'unreadable ' + str(png)
        cv2.imwrite(os.path.join(ROOT, 'images', split, png.stem + '.jpg'),
                    img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        fh = open(os.path.join(ROOT, 'labels', split, png.stem + '.txt'), 'w')
        fh.write('\n'.join(TABLE[(split, png.stem)]) + '\n')
        fh.close()
        return 1

    new = 0
    if cands:
        with ThreadPoolExecutor(max_workers=4) as ex:
            for _ in ex.map(_one, cands):
                new += 1
    shutil.rmtree(src_dir, ignore_errors=True)
    # Flat-disk guarantee: HF blob cache keeps every PNG even after the
    # extracted dir is deleted (~15 GB/folder -> 100 GB wall by folder 4).
    # Purge this repo's blobs now; contract JPGs are the resume mechanism,
    # so the next folder re-downloads clean with zero rework.
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        _repo_cache = os.path.join(HF_HUB_CACHE,
                                   'datasets--' + HF_REPO.replace('/', '--'))
        shutil.rmtree(_repo_cache, ignore_errors=True)
    except Exception as e:
        print('cache purge skipped (%s); watch disk' % e)
    print('done %s/%s: %d new images' % (subset, fld, new))
    return new

for _f in TRAIN_FOLDERS:
    ensure_folder(HF_TRAIN, _f, 'train')
for _f in VAL_FOLDERS:
    ensure_folder(HF_VAL, _f, 'val')
names = ['path: ' + os.path.abspath(ROOT), 'train: images/train',
         'val: images/val', 'test: images/val', 'names:']
names += ['  %d: %s' % (i, n) for i, n in enumerate(CLASSES)]
open(os.path.join(ROOT, 'data.yaml'), 'w').write('\n'.join(names) + '\n')
ntr = len(list(Path(ROOT, 'images', 'train').glob('*.jpg')))
nva = len(list(Path(ROOT, 'images', 'val').glob('*.jpg')))
print('contract: train %d images, val %d images' % (ntr, nva))
assert ntr > 30000 and nva > 9000, 'incomplete rebuild — folders missing?'

## 3. Polish (5 epochs, low LR, full pool)

In [ ]:
assert 'PREV_WEIGHTS' in dir(), 'run cell 0 (Params) first'
import os
from ultralytics import YOLO

pool = list(__import__('pathlib').Path(ROOT, 'images', 'train').glob('*.jpg'))
print('train pool: %d images' % len(pool))
assert len(pool) > 30000, 'train pool incomplete — finish cell 2 first'
print('init weights:', PREV_WEIGHTS)
model = YOLO(PREV_WEIGHTS)
model.train(
    data=os.path.join(ROOT, 'data.yaml'),
    epochs=EPOCHS, imgsz=640, batch=32, patience=8, workers=4,
    lr0=LR0, name='finale',
)  # GPU OOM? batch=16. Host-RAM kill? workers=2.

run_dir = model.trainer.save_dir
BEST = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', BEST)


## 4. Full-val report + per-class table (the publication-grade number)

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Polish) first'
import os
from ultralytics import YOLO

metrics = YOLO(BEST).val(data=os.path.join(ROOT, 'data.yaml'), verbose=False)
names = metrics.names
print('%-10s %9s %9s %9s' % ('class', 'precision', 'recall', 'mAP50'))
per_class = {}
for i, ci in enumerate(metrics.box.ap_class_index):
    nm = names[int(ci)]
    m = round(float(metrics.box.ap50[i]), 4)
    per_class[nm] = m
    print('%-10s %9.3f %9.3f %9.3f' % (nm, float(metrics.box.p[i]), float(metrics.box.r[i]), m))
mAP50 = round(float(metrics.box.map50), 4)
print('FINALE overall mAP50 (full val): %.4f' % mAP50)
print('loop-8 anchor baseline was 0.8293 — delta: %+.4f' % (mAP50 - 0.8293))


## 5. Export + static int8 + final registry

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Polish) first'
import glob, os
import shutil as _sh, traceback
import cv2 as _cv2, numpy as _np
from onnxruntime.quantization import CalibrationDataReader as _CDR, quantize_static as _qs, QuantType as _QT

onnx_path = YOLO(BEST).export(format='onnx', imgsz=640, opset=17, simplify=True)
print('onnx:', onnx_path)

calib_imgs = sorted(glob.glob(os.path.join(ROOT, 'images', 'val', '*.jpg')))[:200]
assert calib_imgs, 'no val images for calibration'

class _FR(_CDR):
    def __init__(self, frames, onx):
        import onnxruntime as _ort
        self.input_name = _ort.InferenceSession(onx, providers=['CPUExecutionProvider']).get_inputs()[0].name
        self.reiter = iter([self._pre(p) for p in frames])
    def _pre(self, path):
        img = _cv2.imread(path)
        img = _cv2.resize(img, (640, 640))
        rgb = _cv2.cvtColor(img, _cv2.COLOR_BGR2RGB).astype(_np.float32) / 255.0
        return {self.input_name: _np.transpose(rgb, (2, 0, 1))[_np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

int8_path = onnx_path.replace('.onnx', '-int8.onnx')
try:
    _qs(onnx_path, int8_path, _FR(calib_imgs, onnx_path), weight_type=_QT.QInt8)
    print('STATIC int8 ->', int8_path)
except Exception:
    traceback.print_exc()
    raise

import datetime, json
reg = os.path.join(ROOT, 'registry', 'india-yolov8n-final')
os.makedirs(reg, exist_ok=True)
_sh.copy(onnx_path, os.path.join(reg, 'model.onnx'))
_sh.copy(int8_path, os.path.join(reg, 'model-int8.onnx'))
meta = {'name': 'india-yolov8n-final', 'classes': CLASSES, 'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255], 'layout': 'NCHW', 'color': 'RGB'},
        'quantization': 'int8', 'source_run': datetime.date.today().isoformat(),
        'metrics': {'mAP50_overall': mAP50, 'per_class_mAP50': per_class},
        'provenance': {'hf_repo': HF_REPO, 'run': 'finale-joint-polish',
                       'init_weights': 'loop-8 best.pt', 'epochs': EPOCHS, 'lr0': LR0,
                       'mapping': 'optionB-merge-6class', 'val': 'official-full-10k'}}
open(os.path.join(reg, 'metadata.json'), 'w').write(json.dumps(meta, indent=2))
print('registry:', reg)


## 6. Finale pack (auto-download — drop this zip in the workspace)

In [ ]:
assert 'BEST' in dir() and 'mAP50' in dir(), 'run cells 3-4 first'
import datetime, os, shutil

pack_name = 'bmd_finale_%s' % datetime.date.today().isoformat()
pack_dir = os.path.join(ROOT, pack_name)
os.makedirs(pack_dir, exist_ok=True)
shutil.copytree(os.path.join(ROOT, 'registry', 'india-yolov8n-final'),
                os.path.join(pack_dir, 'registry'), dirs_exist_ok=True)
shutil.copy2(BEST, os.path.join(pack_dir, 'best.pt'))
results_csv = os.path.join(os.path.dirname(BEST), '..', 'results.csv')
if os.path.isfile(results_csv):
    shutil.copy2(results_csv, pack_dir)
fh = open(os.path.join(pack_dir, 'finale_report.txt'), 'w')
fh.write('finale mAP50 (full val): %.4f\n' % mAP50)
fh.write('per-class: %s\n' % {k: round(v, 4) for k, v in per_class.items()})
fh.write('delta vs loop-8 anchor (0.8293): %+.4f\n' % (mAP50 - 0.8293))
fh.close()
zip_path = shutil.make_archive(os.path.join(ROOT, pack_name), 'zip', root_dir=ROOT, base_dir=pack_name)
print('pack: %s (%.1f MB)' % (zip_path, os.path.getsize(zip_path) / 1e6))
from google.colab import files
files.download(zip_path)
print('DONE — place %s.zip in notebooks/training_output_zips/.' % pack_name)
print('If finale mAP50 < 0.8193: keep loop-8 registry as winner, report it as such.')